# Plant Disease Dataset Preparation

This notebook prepares the `Plant_leave_diseases_dataset_without_augmentation` image dataset for classification.

It will:
- discover class folders and validate image files;
- convert valid images to RGB (3 color channels) and resize them to **256 × 256**;
- assign each image a unique, fixed-length ID;
- place **all processed images in one `train/` directory**;
- create stratified `train.csv` and `test.csv` files with columns `image_name,diagnosis`.

The CSV files determine which images belong to each split. Both CSVs refer to images in the same `train/` directory, as requested. The original dataset is left unchanged.

> **Important:** `train/` contains images from both splits. When evaluating a model, load only filenames listed in `test.csv` for the test set.

## 1. Configure paths and split settings

By default, the notebook expects the dataset folder to be next to this notebook, inside a `data/` directory. Update `DATASET_DIR` if your folder is elsewhere.

The split is stratified by diagnosis, so each class is represented in train and test in approximately the same proportions as in the source dataset.

In [9]:
from pathlib import Path
import random
import shutil
import warnings

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from IPython.display import display

# Change this path if needed.
DATASET_DIR = Path("../data/Plant_leave_diseases_dataset_without_augmentation")

# Output directory. Processed images and CSV files will be written here.
OUTPUT_DIR = Path("../data/plant_disease_processed")

IMAGE_SIZE = (256, 256)
TEST_SIZE = 0.20
RANDOM_STATE = 42

# IDs will be 12 digits, with the file extension .jpg.
ID_WIDTH = 12
JPEG_QUALITY = 95

if not DATASET_DIR.is_dir():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATASET_DIR.resolve()}\n"
        "Update DATASET_DIR in this cell to the correct path."
    )

print("Dataset:", DATASET_DIR.resolve())
print("Output: ", OUTPUT_DIR.resolve())

Dataset: /home/matthiew/projects/Plant-Disease-Diagnosis/data/Plant_leave_diseases_dataset_without_augmentation
Output:  /home/matthiew/projects/Plant-Disease-Diagnosis/data/plant_disease_processed


## 2. Inspect class folders

Each immediate subfolder is treated as a diagnosis label. Files that are not supported image formats are ignored, and unreadable/corrupt images are recorded in a report rather than silently included.

In [4]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

class_dirs = sorted(p for p in DATASET_DIR.iterdir() if p.is_dir())
if not class_dirs:
    raise ValueError(f"No class subfolders found in {DATASET_DIR.resolve()}")

class_summary = []
for class_dir in class_dirs:
    image_files = [
        p for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    class_summary.append({"diagnosis": class_dir.name, "image_files_found": len(image_files)})

class_df = pd.DataFrame(class_summary)
display(class_df)
print(f"Found {len(class_dirs)} class folders.")

,diagnosis,image_files_found
0,Apple___Apple_scab,630
1,Apple___Black_rot,621
2,Apple___Cedar_apple_rust,275
3,Apple___healthy,1645
4,Cherry___Powdery_mildew,1052
5,Cherry___healthy,854
6,Corn___Cercospora_leaf_spot Gray_leaf_spot,513
7,Corn___Common_rust,1192
8,Corn___Northern_Leaf_Blight,985
9,Corn___healthy,1162


Found 33 class folders.


## 3. Read, validate, and standardize images

Every readable image is EXIF-corrected, converted to RGB, and resized to exactly `256 × 256`. The output is saved as JPEG, so each saved image has three color channels and a consistent format. Resizing is direct, so the original aspect ratio may change for non-square images.

In [5]:
# Rebuild output cleanly so rerunning the notebook doesn't leave stale images.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

IMAGES_DIR = OUTPUT_DIR / "train"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

records = []
problems = []
image_counter = 0

for class_dir in class_dirs:
    diagnosis = class_dir.name

    paths = sorted(
        p for p in class_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    )

    for source_path in paths:
        try:
            with Image.open(source_path) as img:
                # Apply camera orientation metadata before resizing.
                img = ImageOps.exif_transpose(img)
                original_format = img.format
                original_mode = img.mode
                original_size = img.size

                # Convert grayscale, palette, RGBA, CMYK, etc. to 3-channel RGB.
                img = img.convert("RGB")

                # PIL expects (width, height), so this produces 256x256.
                if img.size != IMAGE_SIZE:
                    img = img.resize(IMAGE_SIZE, Image.Resampling.LANCZOS)

                image_counter += 1
                image_name = f"{image_counter:0{ID_WIDTH}d}.jpg"
                destination = IMAGES_DIR / image_name
                img.save(destination, format="JPEG", quality=JPEG_QUALITY, optimize=True)

                records.append({
                    "image_name": image_name,
                    "diagnosis": diagnosis,
                    "source_path": str(source_path),
                    "original_width": original_size[0],
                    "original_height": original_size[1],
                    "original_mode": original_mode,
                    "original_format": original_format,
                })

        except (UnidentifiedImageError, OSError, ValueError) as exc:
            problems.append({
                "source_path": str(source_path),
                "diagnosis": diagnosis,
                "error": str(exc),
            })

if not records:
    raise RuntimeError("No readable images were processed. Check DATASET_DIR and the image files.")

manifest = pd.DataFrame(records)
problem_df = pd.DataFrame(problems)

print(f"Processed images: {len(manifest):,}")
print(f"Unreadable/failed images skipped: {len(problem_df):,}")
print(f"Processed image directory: {IMAGES_DIR.resolve()}")

if not problem_df.empty:
    display(problem_df.head(20))

Processed images: 40,000
Unreadable/failed images skipped: 0
Processed image directory: /home/matthiew/projects/Plant-Disease-Diagnosis/data/plant_disease_processed/train


## 4. Verify processed image properties

This checks the actual saved output files, not just the original files. Expected properties are RGB mode, size `256 × 256`, and three channels.

In [6]:
verification_errors = []

for row in manifest.itertuples(index=False):
    path = IMAGES_DIR / row.image_name
    try:
        with Image.open(path) as img:
            if img.size != IMAGE_SIZE:
                verification_errors.append((row.image_name, "wrong dimensions", img.size))
            if img.mode != "RGB":
                verification_errors.append((row.image_name, "not RGB", img.mode))
            if img.format != "JPEG":
                verification_errors.append((row.image_name, "not JPEG", img.format))
            arr = np.asarray(img)
            if arr.shape != (256, 256, 3):
                verification_errors.append((row.image_name, "wrong array shape", arr.shape))
    except Exception as exc:
        verification_errors.append((row.image_name, "could not reopen", str(exc)))

print(f"Images verified: {len(manifest):,}")
print(f"Verification issues: {len(verification_errors):,}")

if verification_errors:
    display(pd.DataFrame(verification_errors, columns=["image_name", "issue", "details"]).head(30))
else:
    print("All processed images are RGB JPEGs with array shape (256, 256, 3).")

Images verified: 40,000
Verification issues: 0
All processed images are RGB JPEGs with array shape (256, 256, 3).


## 5. Create a stratified train/test split

The split uses diagnosis labels for stratification. `train.csv` and `test.csv` each contain exactly two columns:

- `image_name`: the unique filename inside `train/`
- `diagnosis`: the class label from the source folder name

A class needs at least two readable images to appear in both splits. The code reports any classes too small for a stratified split and stops with a helpful error instead of silently creating a misleading split.

In [7]:
class_counts = manifest["diagnosis"].value_counts().sort_index()
too_small = class_counts[class_counts < 2]

if not too_small.empty:
    raise ValueError(
        "Stratified train/test split requires at least 2 readable images per class. "
        f"Classes with fewer than 2 images: {too_small.to_dict()}"
    )

train_df, test_df = train_test_split(
    manifest[["image_name", "diagnosis"]],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=manifest["diagnosis"],
    shuffle=True,
)

# Keep the requested column order and make row order deterministic.
train_df = train_df[["image_name", "diagnosis"]].sort_values("image_name").reset_index(drop=True)
test_df = test_df[["image_name", "diagnosis"]].sort_values("image_name").reset_index(drop=True)

train_csv = OUTPUT_DIR / "train.csv"
test_csv = OUTPUT_DIR / "test.csv"

train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)

print(f"Train rows: {len(train_df):,} ({len(train_df) / len(manifest):.1%})")
print(f"Test rows:  {len(test_df):,} ({len(test_df) / len(manifest):.1%})")
print("Saved:", train_csv.resolve())
print("Saved:", test_csv.resolve())

display(train_df.head())
display(test_df.head())

Train rows: 32,000 (80.0%)
Test rows:  8,000 (20.0%)
Saved: /home/matthiew/projects/Plant-Disease-Diagnosis/data/plant_disease_processed/train.csv
Saved: /home/matthiew/projects/Plant-Disease-Diagnosis/data/plant_disease_processed/test.csv


,image_name,diagnosis
0,000000000001.jpg,Apple___Apple_scab
1,000000000003.jpg,Apple___Apple_scab
2,000000000004.jpg,Apple___Apple_scab
3,000000000006.jpg,Apple___Apple_scab
4,000000000007.jpg,Apple___Apple_scab


,image_name,diagnosis
0,000000000002.jpg,Apple___Apple_scab
1,000000000005.jpg,Apple___Apple_scab
2,000000000014.jpg,Apple___Apple_scab
3,000000000015.jpg,Apple___Apple_scab
4,000000000021.jpg,Apple___Apple_scab


## 6. Check split integrity and class balance

The checks below confirm that filenames are unique, train and test have no overlapping rows, all CSV references point to existing images, and both splits contain the expected classes.

In [8]:
assert list(train_df.columns) == ["image_name", "diagnosis"]
assert list(test_df.columns) == ["image_name", "diagnosis"]
assert train_df["image_name"].is_unique
assert test_df["image_name"].is_unique
assert set(train_df["image_name"]).isdisjoint(set(test_df["image_name"]))
assert set(train_df["image_name"]).union(set(test_df["image_name"])) == set(manifest["image_name"])
assert all((IMAGES_DIR / name).is_file() for name in train_df["image_name"])
assert all((IMAGES_DIR / name).is_file() for name in test_df["image_name"])

train_classes = set(train_df["diagnosis"])
test_classes = set(test_df["diagnosis"])
all_classes = set(manifest["diagnosis"])

print("Integrity checks passed.")
print("All classes in train:", train_classes == all_classes)
print("All classes in test: ", test_classes == all_classes)

balance = (
    pd.concat([
        train_df["diagnosis"].value_counts().rename("train_count"),
        test_df["diagnosis"].value_counts().rename("test_count"),
    ], axis=1)
    .fillna(0)
    .astype(int)
    .sort_index()
)
balance["total"] = balance["train_count"] + balance["test_count"]
balance["test_fraction"] = (balance["test_count"] / balance["total"]).round(3)
display(balance)

Integrity checks passed.
All classes in train: True
All classes in test:  True


,train_count,test_count,total,test_fraction
diagnosis,,,,
Apple___Apple_scab,504,126,630,0.200
Apple___Black_rot,497,124,621,0.200
Apple___Cedar_apple_rust,220,55,275,0.200
Apple___healthy,1316,329,1645,0.200
Cherry___Powdery_mildew,842,210,1052,0.200
Cherry___healthy,683,171,854,0.200
Corn___Cercospora_leaf_spot Gray_leaf_spot,410,103,513,0.201
Corn___Common_rust,954,238,1192,0.200
Corn___Northern_Leaf_Blight,788,197,985,0.200


## 7. Final output structure

After running all cells, the output directory will look like this:

```text
plant_disease_processed/
├── train/
│   ├── 000000000001.jpg
│   ├── 000000000002.jpg
│   └── ...
├── train.csv
└── test.csv
```

Example CSV rows:

```csv
image_name,diagnosis
000000000001.jpg,Apple___Apple_scab
000000000002.jpg,Tomato___healthy
```

To load a CSV and image later:

```python
from pathlib import Path
import pandas as pd
from PIL import Image

root = Path("../data/plant_disease_processed")
train_df = pd.read_csv(root / "train.csv")

row = train_df.iloc[0]
image = Image.open(root / "train" / row["image_name"]).convert("RGB")
label = row["diagnosis"]
```

**Dataset hygiene note:** This split is stratified, but it does not detect near-duplicate or duplicate images across the original dataset. If duplicates exist, they can make test performance look better than it really is.